# AgenticSZZ Run Analysis

Where the agent succeeds, where it fails, and whether its failures are a matter of capability or of output format. The run is a repeated evaluation of `attribution-agentic-szz-evaluation`, the knowledge graph driven attribution, scored against the defect commits recorded in the validation context index. It mirrors the analysis of the repository driven run, so the two are read side by side.

A prediction counts as correct when it prefixes one of the recorded defect commits, or is prefixed by one, from seven characters upward. Each trace falls into exactly one of four outcomes.

| Outcome | Meaning |
|---|---|
| `correct` | the predicted commit is one of the recorded defects |
| `incorrect` | a commit was predicted and it is not one of them |
| `empty` | no commit reached the output field |
| `error` | the graph itself failed, for instance on a recursion limit |

Cases carry the category of their defect within the knowledge graph the agent traverses. The category says how far the defect sits from the correction.

| Category | Meaning |
|---|---|
| `blame` | the defect is the commit that git blame reaches directly |
| `blame-ancestor` | the defect lies above the blame result in its history |
| `correction-ancestor` | the defect lies in the ancestry of the correction |
| `outside` | the defect appears in no commit of the index at all |

## Inputs

The evaluation traces hold one event stream per run. The context index holds the recorded defect commits and the commit pool of each case, from which the categories are taken. Loading the traces pulls the whole event stream into memory at once and has crashed the kernel before.

In [1]:
import io
import json
import re
from pathlib import Path

import polars as pl

from benchmark.attribution.evaluation import (
    MessageProcessor,
    TokenProcessor,
    TraceSerializer,
)
from benchmark.tracing import Trace
from benchmark_notebook.io import CloudStorage

In [2]:
traces = pl.from_arrow(
    CloudStorage(bucket="bug-request-bucket").asset(
        key="attribution-agentic-szz-evaluation",
        version="1.0.0",
        location="main",
        partition="default",
    )
)

index = pl.from_arrow(
    CloudStorage(bucket="bug-request-bucket").asset(
        key="ingest-validation-context-index",
        version="2.0.0",
        location="main",
        partition="default",
    )
)

In [3]:
def _extract_introducer(payload: str) -> str | None:
    p = json.loads(payload)
    return p.get("outputs", {}).get("introducer")


def _extract_error(payload: str) -> str | None:
    p = json.loads(payload)
    return p.get("error")


# Root LangGraph node_end events carry the prediction
predictions_end = (
    traces.filter(
        (pl.col("node") == "LangGraph")
        & (pl.col("event") == "node_end")
        & pl.col("parent").is_null()
    )
    .with_columns(
        pl.col("payload")
        .map_elements(_extract_introducer, return_dtype=pl.String)
        .alias("predicted"),
    )
    .select(["trace", "repository", "correction", "predicted"])
    .with_columns(pl.lit(None).cast(pl.String).alias("error"))
)

# Root LangGraph node_error events -- graph crashed (e.g. recursion limit)
predictions_error = (
    traces.filter(
        (pl.col("node") == "LangGraph")
        & (pl.col("event") == "node_error")
        & pl.col("parent").is_null()
    )
    .with_columns(
        pl.lit(None).cast(pl.String).alias("predicted"),
        pl.col("payload")
        .map_elements(_extract_error, return_dtype=pl.String)
        .alias("error"),
    )
    .select(["trace", "repository", "correction", "predicted", "error"])
)

predictions = pl.concat([predictions_end, predictions_error])

# Deduplicate: keep node_end over node_error when both exist for the same trace
predictions = predictions.sort("error", nulls_last=False).group_by("trace").first()
predictions.shape

(3017, 5)

In [4]:
def _match(predicted: str | None, defects: list[str]) -> bool:
    if predicted is None:
        return False
    p = predicted.lower().strip()
    if len(p) < 7:
        return False
    return any(d.lower().startswith(p) or p.startswith(d.lower()) for d in defects)


ground = index.select(["repository", "correction", "defects"])
joined = predictions.join(ground, on=["repository", "correction"], how="left")

joined = joined.with_columns(
    pl.cum_count("trace").over(["repository", "correction"]).alias("run")
)

evaluated = joined.with_columns(
    pl.struct(["predicted", "defects"])
    .map_elements(
        lambda row: _match(row["predicted"], row["defects"] or []),
        return_dtype=pl.Boolean,
    )
    .alias("correct"),
    (
        pl.col("predicted").is_not_null() & (pl.col("predicted").str.len_chars() >= 7)
    ).alias("has_prediction"),
)

# -- Outcome: correct / incorrect / error / empty (mutually exclusive) --
evaluated = evaluated.with_columns(
    pl.when(pl.col("error").is_not_null())
    .then(pl.lit("error"))
    .when(pl.col("correct"))
    .then(pl.lit("correct"))
    .when(pl.col("has_prediction"))
    .then(pl.lit("incorrect"))
    .otherwise(pl.lit("empty"))
    .alias("outcome")
)

# -- Join category from commits (the category of the ground-truth defect) --
defect_categories = (
    pl.from_arrow(index)
    .select(["repository", "correction", "defects", "commits"])
    .explode("defects")
    .explode("commits")
    .drop_nulls("commits")
    .unnest("commits")
    .filter(pl.col("commit") == pl.col("defects"))
    .group_by(["repository", "correction"])
    .agg(pl.col("category").first())
    .with_columns(pl.col("category").str.replace_all("_", "-"))
)

# -- Join language group (py, js, py-js) --
lang_groups = (
    pl.from_arrow(index)
    .select(["repository", "correction", "languages"])
    .with_columns(
        pl.when(
            pl.col("languages").list.contains("py")
            & pl.col("languages").list.contains("js")
        )
        .then(pl.lit("py-js"))
        .when(pl.col("languages").list.contains("py"))
        .then(pl.lit("py"))
        .when(pl.col("languages").list.contains("js"))
        .then(pl.lit("js"))
        .otherwise(pl.lit("other"))
        .alias("language")
    )
    .select(["repository", "correction", "language"])
)

evaluated = (
    evaluated.join(defect_categories, on=["repository", "correction"], how="left")
    .join(lang_groups, on=["repository", "correction"], how="left")
    .with_columns(pl.col("category").fill_null("outside"))
)
evaluated.group_by("outcome").len().sort("outcome")

outcome,len
str,u32
"""correct""",2176
"""empty""",51
"""error""",113
"""incorrect""",677


## Case Breakdown

In [5]:
def _pct(n, total):
    pct = 100 * n / total if total > 0 else 0.0
    return f"{pct:.1f}% ({n})"


def _breakdown(frame, group_col=None):
    if group_col:
        raw = (
            frame.group_by(group_col)
            .agg(
                pl.len().alias("total"),
                pl.col("correct").sum().alias("correct"),
                (pl.col("has_prediction") & ~pl.col("correct"))
                .sum()
                .alias("incorrect"),
                (~pl.col("has_prediction")).sum().alias("no_answer"),
                pl.col("error").is_not_null().sum().alias("errors"),
            )
            .sort(group_col)
        )
    else:
        raw = frame.select(
            pl.len().alias("total"),
            pl.col("correct").sum().alias("correct"),
            (pl.col("has_prediction") & ~pl.col("correct")).sum().alias("incorrect"),
            (~pl.col("has_prediction")).sum().alias("no_answer"),
            pl.col("error").is_not_null().sum().alias("errors"),
        )

    rows = []
    for r in raw.iter_rows(named=True):
        t = r["total"]
        row = {
            "total": t,
            "correct": _pct(r["correct"], t),
            "incorrect": _pct(r["incorrect"], t),
            "no_answer": _pct(r["no_answer"], t),
            "errors": _pct(r["errors"], t),
        }
        if group_col:
            row[group_col] = r[group_col]
        rows.append(row)

    cols = [group_col] if group_col else []
    cols += ["total", "correct", "incorrect", "no_answer", "errors"]
    return pl.DataFrame(rows).select(cols)


print("=== Overall ===")
display(_breakdown(evaluated))

print("\n=== By Category ===")
display(_breakdown(evaluated, "category"))

print("\n=== By Language ===")
display(_breakdown(evaluated, "language"))

=== Overall ===


total,correct,incorrect,no_answer,errors
i64,str,str,str,str
3017,"""72.1% (2176)""","""22.4% (677)""","""5.4% (164)""","""3.7% (113)"""



=== By Category ===


category,total,correct,incorrect,no_answer,errors
str,i64,str,str,str,str
"""blame""",2298,"""87.3% (2006)""","""8.1% (186)""","""4.6% (106)""","""3.6% (83)"""
"""blame-ancestor""",96,"""34.4% (33)""","""59.4% (57)""","""6.2% (6)""","""1.0% (1)"""
"""correction-ancestor""",444,"""30.2% (134)""","""62.2% (276)""","""7.7% (34)""","""3.8% (17)"""
"""outside""",179,"""1.7% (3)""","""88.3% (158)""","""10.1% (18)""","""6.7% (12)"""



=== By Language ===


language,total,correct,incorrect,no_answer,errors
str,i64,str,str,str,str
"""py""",3017,"""72.1% (2176)""","""22.4% (677)""","""5.4% (164)""","""3.7% (113)"""


## Incorrect Predictions by Predicted Category

For each incorrect trace: which category does the **predicted** SHA fall
into within the same case's commit pool? This is distinct from the
ground-truth category of the case (above) -- it asks where in the
knowledge-graph the model's wrong answer landed.

In [6]:
# Long table of (repository, correction, commit, category) for every commit in the index
commits_long = (
    pl.from_arrow(index)
    .select(["repository", "correction", "commits"])
    .explode("commits")
    .drop_nulls("commits")
    .unnest("commits")
    .select(["repository", "correction", "commit", "category"])
    .with_columns(pl.col("category").str.replace_all("_", "-"))
)

incorrect = (
    evaluated.filter(pl.col("outcome") == "incorrect")
    .select(["trace", "repository", "correction", "predicted", "category"])
    .rename({"category": "truth_category"})
)

# Match predicted SHA to a commit in the same case via prefix on either side
incorrect_pool = incorrect.join(
    commits_long, on=["repository", "correction"], how="left"
).with_columns(
    pl.col("predicted").str.to_lowercase().alias("_p"),
    pl.col("commit").str.to_lowercase().alias("_c"),
)
incorrect_matched = (
    incorrect_pool.filter(
        pl.col("_c").str.starts_with(pl.col("_p"))
        | pl.col("_p").str.starts_with(pl.col("_c"))
    )
    .group_by("trace")
    .agg(pl.col("category").first().alias("predicted_category"))
)

incorrect_classified = incorrect.join(
    incorrect_matched, on="trace", how="left"
).with_columns(pl.col("predicted_category").fill_null("unknown"))

print(f"=== Incorrect predictions: {incorrect_classified.height} ===")

print("\n=== Distribution by predicted category ===")
display(
    incorrect_classified.group_by("predicted_category")
    .agg(pl.len().alias("n"))
    .with_columns(
        (pl.col("n") / pl.lit(incorrect_classified.height) * 100)
        .round(1)
        .alias("pct")
    )
    .sort("n", descending=True)
)

print("\n=== Predicted category x truth category ===")
display(
    incorrect_classified.group_by(["truth_category", "predicted_category"])
    .agg(pl.len().alias("n"))
    .sort(["truth_category", "predicted_category"])
    .pivot(on="predicted_category", index="truth_category", values="n")
    .fill_null(0)
    .sort("truth_category")
)

=== Incorrect predictions: 677 ===

=== Distribution by predicted category ===


predicted_category,n,pct
str,u32,f64
"""blame""",561,82.9
"""blame-ancestor""",63,9.3
"""correction-ancestor""",29,4.3
"""correction""",18,2.7
"""unknown""",6,0.9



=== Predicted category x truth category ===


truth_category,blame,blame-ancestor,correction,correction-ancestor,unknown
str,u32,u32,u32,u32,u32
"""blame""",134,29,13,5,5
"""blame-ancestor""",50,6,1,0,0
"""correction-ancestor""",236,17,3,19,1
"""outside""",141,11,1,5,0


## OTLP Export for No-Answer Cases

In [7]:
no_answer_cases = evaluated.filter(~pl.col("has_prediction"))
no_answer_keys = no_answer_cases.select(
    ["trace", "repository", "correction", "run", "error", "defects"]
).unique()

print(f"{no_answer_keys.height} traces without a parseable prediction.")
no_answer_keys

164 traces without a parseable prediction.


trace,repository,correction,run,error,defects
str,str,str,u32,str,list[str]
"""d9cdcf0085f2446c81f343729eefc4…","""dnouri/deform""","""65f05b3d98536f9af123e0ec8b38fc…",7,"""GumTree parse error side=src t…","[""5b5f89a80b42d5fd5d343f4bbedd4b5c3624e3e8""]"
"""48f8f10d3f5946c38282c56ee5ea6b…","""floodlight/oftest""","""c41fbece667088409d6ebce617f5ac…",10,"""GumTree parse error side=src t…","[""e4b384df950b553313a628ef9c37cba455ba8243""]"
"""cffa5c7ea72d445bb830bbc456b832…","""terminalmage/salt""","""c8a5decdf10a6be46631b0b92742c7…",4,null,"[""17b4538567de730ebd163f706c3880cbe4251715""]"
"""c43df296f1184f44be9469a22d9e95…","""floodlight/oftest""","""c41fbece667088409d6ebce617f5ac…",6,"""GumTree parse error side=src t…","[""e4b384df950b553313a628ef9c37cba455ba8243""]"
"""4095b973f0164aad93e33ed4e0bf33…","""PierreRaybaut/guiqwt""","""685112981b32686d853f3183fa3d5b…",10,null,"[""851d1950ae9d319e19075f81a1cafde1ad077b4f""]"
…,…,…,…,…,…
"""2c9535224dce41bab1b7cda36e9014…","""dnouri/deform""","""65f05b3d98536f9af123e0ec8b38fc…",10,"""GumTree parse error side=src t…","[""5b5f89a80b42d5fd5d343f4bbedd4b5c3624e3e8""]"
"""a0654ce5bae840a59c4a7f610f6098…","""markcwill/obspy""","""c5b35452475ed915c7d5ce592ef9a3…",7,"""Could not parse response conte…","[""4e70c059e8f76d3e71da45d31ac31b6759f7b4b2""]"
"""ced23530d4264fccbfa658bf25c3b1…","""floodlight/oftest""","""c41fbece667088409d6ebce617f5ac…",9,"""GumTree parse error side=src t…","[""e4b384df950b553313a628ef9c37cba455ba8243""]"


In [8]:
output_dir = Path(".traces") / "agentic-no-answer"

otlp_paths = []

for row in no_answer_keys.iter_rows(named=True):
    trace_id = row["trace"]
    repo = row["repository"]
    correction = row["correction"]

    trace_events = traces.filter(pl.col("trace") == trace_id).sort("sequence")
    trace_obj = Trace(
        seed=trace_id,
        metadata={
            "repository": repo,
            "correction": correction,
        },
    )
    for event in trace_events.iter_rows(named=True):
        trace_obj._events.append(
            {
                "trace": event["trace"],
                "sequence": event["sequence"],
                "timestamp": event["timestamp"],
                "node": event["node"],
                "span": event["span"],
                "parent": event["parent"],
                "event": event["event"],
                "payload": event["payload"],
            }
        )
    trace_obj._sequence = trace_events.height

    dest = output_dir / repo / f"{correction[:12]}.otlp.json"
    dest.parent.mkdir(parents=True, exist_ok=True)

    serializer = TraceSerializer(
        processors=[
            MessageProcessor(),
            TokenProcessor(),
        ]
    )
    resource = {
        "service.name": "AgenticSZZ",
        "repository": repo,
        "correction": correction,
    }
    buf = io.StringIO()
    serializer.serialize(trace_obj, buf, resource=resource)
    dest.write_text(buf.getvalue(), encoding="utf-8")
    otlp_paths.append(
        {
            "repository": repo,
            "correction": correction,
            "path": str(dest),
            "events": trace_events.height,
        }
    )

otlp_index = pl.DataFrame(otlp_paths)
otlp_index

repository,correction,path,events
str,str,str,i64
"""dnouri/deform""","""65f05b3d98536f9af123e0ec8b38fc…",""".traces/agentic-no-answer/dnou…",6
"""floodlight/oftest""","""c41fbece667088409d6ebce617f5ac…",""".traces/agentic-no-answer/floo…",6
"""terminalmage/salt""","""c8a5decdf10a6be46631b0b92742c7…",""".traces/agentic-no-answer/term…",106
"""floodlight/oftest""","""c41fbece667088409d6ebce617f5ac…",""".traces/agentic-no-answer/floo…",6
"""PierreRaybaut/guiqwt""","""685112981b32686d853f3183fa3d5b…",""".traces/agentic-no-answer/Pier…",172
…,…,…,…
"""dnouri/deform""","""65f05b3d98536f9af123e0ec8b38fc…",""".traces/agentic-no-answer/dnou…",6
"""markcwill/obspy""","""c5b35452475ed915c7d5ce592ef9a3…",""".traces/agentic-no-answer/mark…",86
"""floodlight/oftest""","""c41fbece667088409d6ebce617f5ac…",""".traces/agentic-no-answer/floo…",6


## Hidden Predictions in No-Answer Traces

Some traces produce no parseable `introducer` SHA but the LLM may
still mention a commit hash or author attribution in its final
message. We scan the last LLM generation in each no-answer trace
for hex strings >= 7 characters that look like SHA prefixes.

In [9]:
SHA_PATTERN = re.compile(r"\b[0-9a-f]{7,40}\b")

hidden = []

for row in no_answer_keys.iter_rows(named=True):
    trace_id = row["trace"]
    trace_events = traces.filter(pl.col("trace") == trace_id).sort("sequence")

    llm_ends = trace_events.filter(pl.col("event") == "llm_end")
    if llm_ends.height == 0:
        hidden.append(
            {
                "trace": trace_id,
                "repository": row["repository"],
                "correction": row["correction"],
                "error": row["error"],
                "defects": row["defects"],
                "last_generation": None,
                "sha_candidates": [],
                "has_hidden_sha": False,
                "form": "no_llm_output",
            }
        )
        continue

    last_llm = llm_ends.row(-1, named=True)
    payload = json.loads(last_llm["payload"])

    generation_text = ""
    for batch in payload.get("generations") or []:
        for g in batch if isinstance(batch, list) else [batch]:
            if not isinstance(g, dict):
                continue
            text = g.get("text") or ""
            generation_text += text
            msg = g.get("message") or {}
            if isinstance(msg, dict):
                content = msg.get("content") or ""
                if isinstance(content, str):
                    generation_text += " " + content
                for call in msg.get("tool_calls") or []:
                    args = json.dumps(call.get("args", {}))
                    generation_text += " " + args

    sha_matches = SHA_PATTERN.findall(generation_text.lower())

    # Also check the root node payload for partial outputs
    root_ends = trace_events.filter(
        (pl.col("node") == "LangGraph")
        & (pl.col("event").is_in(["node_end", "node_error"]))
        & pl.col("parent").is_null()
    )
    if root_ends.height > 0:
        root_payload = json.loads(root_ends.row(-1, named=True)["payload"])
        sha_matches += SHA_PATTERN.findall(json.dumps(root_payload).lower())

    sha_candidates = sorted(set(sha_matches))

    if row["error"]:
        form = "error"
    elif not generation_text.strip():
        form = "empty_generation"
    elif sha_candidates:
        form = "has_sha_in_text"
    else:
        form = "no_sha_at_all"

    snippet = generation_text[-500:] if len(generation_text) > 500 else generation_text

    hidden.append(
        {
            "trace": trace_id,
            "repository": row["repository"],
            "correction": row["correction"],
            "error": row["error"],
            "defects": row["defects"],
            "last_generation": snippet,
            "sha_candidates": sha_candidates,
            "has_hidden_sha": bool(sha_candidates),
            "form": form,
        }
    )

hidden_df = pl.DataFrame(hidden)
form_counts = (
    hidden_df.group_by("form")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)
form_counts

form,count
str,u32
"""no_llm_output""",73
"""has_sha_in_text""",51
"""error""",40


In [10]:
with_sha = hidden_df.filter(pl.col("has_hidden_sha"))
with_sha.select(
    [
        "repository",
        "correction",
        "form",
        "sha_candidates",
        "defects",
        "last_generation",
    ]
)

repository,correction,form,sha_candidates,defects,last_generation
str,str,str,list[str],list[str],str
"""terminalmage/salt""","""c8a5decdf10a6be46631b0b92742c7…","""has_sha_in_text""","[""183999999994"", ""c7d3ac58efcda4178787cd81cfce9cabc1e797fe"", ""c8a5decdf10a6be46631b0b92742c73ab9eecb1f""]","[""17b4538567de730ebd163f706c3880cbe4251715""]","""{""introducer"": null}"""
"""PierreRaybaut/guiqwt""","""685112981b32686d853f3183fa3d5b…","""has_sha_in_text""","[""685112981b32686d853f3183fa3d5badacfecae5"", ""cf8917024ccf07ef2415a7e44dcf62803954177d""]","[""851d1950ae9d319e19075f81a1cafde1ad077b4f""]","""{""introducer"": null}"""
"""ArduPilot/ardupilot""","""b191269cc82050085b6900fca7a20f…","""has_sha_in_text""","[""155000000002"", ""8fb6ccc8b79f1a81cd6db93390dd0a6a1bef38a9"", ""b191269cc82050085b6900fca7a20fc013cff02d""]","[""06a515645e3390c6f5fbbdda1c62aa3aebfd5cda""]","""{ ""introducer"": ""tridge@gmai…"
"""gaborpapp/AIam""","""39b5ff49bbdf6639b3ddca8cce5eb3…","""error""","[""943999999996"", ""978fa6f6b27fdf0947266b82780bc26ed3d3040"", ""a0c2b985e8e30bc7""]","[""978fa6f6b27fdf0947266b82780bc26ed63d3040""]",""" {""path"": ""movement_ai/applic…"
"""praet/sluk""","""0bd44a41672fc47047913c0a0b0c8a…","""has_sha_in_text""","[""0bd44a41672fc47047913c0a0b0c8a6fb8448413"", ""181999999997"", ""bbaab23f33678f567b7f7c6df120072fd628cf08""]","[""537dc59c7c922a0b7f4d1a2732d171076dfcf05f""]","""{""introducer"": null}"""
…,…,…,…,…,…
"""terminalmage/salt""","""c8a5decdf10a6be46631b0b92742c7…","""has_sha_in_text""","[""42000000001"", ""c7d3ac58efcda4178787cd81cfce9cabc1e797fe"", ""c8a5decdf10a6be46631b0b92742c73ab9eecb1f""]","[""17b4538567de730ebd163f706c3880cbe4251715""]","""{""introducer"": null}"""
"""t2y/python-study""","""4417a94e8c1e6e286d6a2c0ac18fc8…","""has_sha_in_text""","[""38100000001"", ""4417a94e8c1e6e286d6a2c0ac18fc8c6dca052e2"", ""8973df6f2c871620ecf6768b839d7bfcb7cb4bfd""]","[""8973df6f2c871620ecf6768b839d7bfcb7cb4bfd""]","""{ ""introducer"": ""Refactor de…"
"""markcwill/obspy""","""c5b35452475ed915c7d5ce592ef9a3…","""error""","[""31100000002""]","[""4e70c059e8f76d3e71da45d31ac31b6759f7b4b2""]",""" """


## Stratified Accuracy

| Level | Description |
|-------|-------------|
| **Overall** | All cases (no-answer counts as wrong) |
| **Trace exists** | Only cases where the root LangGraph span completed without error |
| **Has SHA-like** | Only cases where `introducer` is a hex string >= 7 chars |

In [11]:
def _accuracy(frame, label):
    n = frame.height
    correct = frame.filter(pl.col("correct")).height
    return {
        "level": label,
        "n": n,
        "correct": correct,
        "accuracy": correct / n if n > 0 else 0.0,
    }


def _stratified(frame, label_prefix=""):
    error_traces_local = hidden_df.filter(pl.col("form") == "error").select("trace")

    rows = []
    rows.append(_accuracy(frame, f"{label_prefix}Overall"))

    trace_ok = frame.join(error_traces_local, on="trace", how="anti")
    rows.append(_accuracy(trace_ok, f"{label_prefix}Trace exists (no error)"))

    has_sha = frame.filter(pl.col("has_prediction"))
    rows.append(_accuracy(has_sha, f"{label_prefix}Has SHA-like prediction"))
    return rows


# -- Overall --
print("=== Overall ===")
display(pl.DataFrame(_stratified(evaluated)))

# -- By Category --
print("\n=== By Category ===")
cat_rows = []
for cat in sorted(evaluated["category"].drop_nulls().unique().to_list()):
    cat_rows.extend(
        _stratified(evaluated.filter(pl.col("category") == cat), f"[{cat}] ")
    )
display(pl.DataFrame(cat_rows))

# -- By Language --
print("\n=== By Language ===")
lang_rows = []
for lang in sorted(evaluated["language"].drop_nulls().unique().to_list()):
    lang_rows.extend(
        _stratified(evaluated.filter(pl.col("language") == lang), f"[{lang}] ")
    )
display(pl.DataFrame(lang_rows))

=== Overall ===


level,n,correct,accuracy
str,i64,i64,f64
"""Overall""",3017,2176,0.721246
"""Trace exists (no error)""",2977,2176,0.730937
"""Has SHA-like prediction""",2853,2176,0.762706



=== By Category ===


level,n,correct,accuracy
str,i64,i64,f64
"""[blame] Overall""",2298,2006,0.872933
"""[blame] Trace exists (no error…",2276,2006,0.881371
"""[blame] Has SHA-like predictio…",2192,2006,0.915146
"""[blame-ancestor] Overall""",96,33,0.34375
"""[blame-ancestor] Trace exists …",95,33,0.347368
…,…,…,…
"""[correction-ancestor] Trace ex…",439,134,0.305239
"""[correction-ancestor] Has SHA-…",410,134,0.326829
"""[outside] Overall""",179,3,0.01676



=== By Language ===


level,n,correct,accuracy
str,i64,i64,f64
"""[py] Overall""",3017,2176,0.721246
"""[py] Trace exists (no error)""",2977,2176,0.730937
"""[py] Has SHA-like prediction""",2853,2176,0.762706


## Tool Usage vs Outcome

Generic breakdown over all `tool_end` events grouped by tool node,
since the AgenticSZZ pipeline invokes a different tool inventory
(knowledge-graph queries) than the repository-SZZ pipeline.

In [12]:
tool_events = traces.filter(pl.col("event") == "tool_end")

# (a) Tool inventory: which tools fired and how often
print("=== Tool inventory ===")
display(
    tool_events.group_by("node")
    .agg(
        pl.len().alias("calls"),
        pl.col("trace").n_unique().alias("traces"),
    )
    .sort("calls", descending=True)
)

# (b) Total tool calls per outcome
tool_counts = tool_events.group_by("trace").agg(pl.len().alias("tool_calls"))
eval_tools = evaluated.join(tool_counts, on="trace", how="left").with_columns(
    pl.col("tool_calls").fill_null(0)
)

print("\n=== Total tool calls by outcome ===")
display(
    eval_tools.group_by("outcome")
    .agg(
        pl.len().alias("n"),
        pl.col("tool_calls").mean().alias("mean"),
        pl.col("tool_calls").median().alias("median"),
        pl.col("tool_calls").min().alias("min"),
        pl.col("tool_calls").max().alias("max"),
        pl.col("tool_calls").sum().alias("total"),
    )
    .sort("outcome")
)

# (c) Per-tool calls per outcome
per_tool = tool_events.group_by(["trace", "node"]).agg(pl.len().alias("calls"))
per_tool_outcome = per_tool.join(
    evaluated.select(["trace", "outcome"]), on="trace", how="inner"
)

print("\n=== Per-tool calls by outcome (totals) ===")
if per_tool_outcome.height > 0:
    display(
        per_tool_outcome.group_by(["node", "outcome"])
        .agg(pl.col("calls").sum().alias("calls"))
        .sort(["node", "outcome"])
        .pivot(on="outcome", index="node", values="calls")
        .fill_null(0)
        .sort("node")
    )
else:
    print("(no tool_end events in this run)")

=== Tool inventory ===


node,calls,traces
str,u32,u32
"""content""",6152,2937
"""candidates""",2945,2943
"""meta""",881,668
"""neighbors""",241,216



=== Total tool calls by outcome ===


outcome,n,mean,median,min,max,total
str,u32,f64,f64,u32,u32,u32
"""correct""",2176,3.298713,3.0,1,14,7178
"""empty""",51,4.078431,4.0,2,7,208
"""error""",113,1.070796,0.0,0,5,121
"""incorrect""",677,4.005908,4.0,1,15,2712



=== Per-tool calls by outcome (totals) ===


node,correct,empty,error,incorrect
str,u32,u32,u32,u32
"""candidates""",2175,53,40,677
"""content""",4330,134,76,1612
"""meta""",580,13,4,284
"""neighbors""",93,8,1,139
